# 03. Build Frontend JSON

TRIBE raw output을 읽어 프론트가 바로 소비할 수 있는 `{date}.json`, `{date}-transcript.json`을 생성한다.

- `attention_proxy`: 세그먼트 반응 강도와 변화율
- `load_proxy`: 반응 지속도와 텍스트 밀도
- `novelty_proxy`: 직전 세그먼트 대비 변화량

In [ ]:
from google.colab import drive
from pathlib import Path
import csv
import json
import numpy as np

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/tribe-v2-student-reaction')
PREPARED_DIR = ROOT / 'outputs' / 'prepared'
RAW_DIR = ROOT / 'outputs' / 'raw'
ASSET_DIR = ROOT / 'outputs' / 'assets'
FRONTEND_DIR = ROOT / 'outputs' / 'frontend'
METADATA_CSV = ROOT / 'inputs' / 'metadata' / '강의 메타데이터.csv'

FRONTEND_DIR.mkdir(parents=True, exist_ok=True)
PILOT_DATES = ['2026-02-02', '2026-02-09', '2026-02-24']

In [ ]:
def clamp(value, lo=0.0, hi=100.0):
    return max(lo, min(hi, value))

def normalize(values):
    values = np.asarray(values, dtype='float32')
    lo = float(values.min())
    hi = float(values.max())
    if hi - lo < 1e-6:
        return np.full_like(values, 50.0)
    return ((values - lo) / (hi - lo) * 100.0).astype('float32')

def labels_for(attention, load, novelty):
    labels = []
    if attention >= 70:
        labels.append('집중 상승')
    elif attention <= 42:
        labels.append('집중 하락')
    if load >= 68:
        labels.append('부하 높음')
    elif novelty <= 35:
        labels.append('복습 필요')
    return labels[:2] or ['리듬 안정']

def interpretation_for(attention, load, novelty):
    if attention >= 72 and novelty >= 58:
        return '새 개념 전환이나 예시 도입으로 반응이 함께 올라가는 구간으로 읽힙니다.'
    if load >= 70 and attention < 55:
        return '설명 밀도에 비해 반응 유지가 낮아 인지 부하가 높게 읽히는 구간입니다.'
    if attention < 45:
        return '반응 강도가 낮아지는 구간입니다. 질문이나 리캡 장치가 필요할 수 있습니다.'
    return '반응이 큰 흔들림 없이 유지되는 안정 구간입니다.'

def load_metadata():
    by_date = {}
    with open(METADATA_CSV, encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            if row['date'] not in by_date:
                by_date[row['date']] = {
                    'subject': row['subject'],
                    'content': row['content'],
                    'instructor': row['instructor'],
                }
    return by_date

metadata_by_date = load_metadata()

In [ ]:
for date in PILOT_DATES:
    prepared_segments = json.loads((PREPARED_DIR / date / 'segments.json').read_text(encoding='utf-8'))
    transcript_payload = json.loads((PREPARED_DIR / date / 'transcript.json').read_text(encoding='utf-8'))
    raw_preds = np.load(RAW_DIR / f'{date}-tribe-raw.npz')['preds']
    color_payload = json.loads((ASSET_DIR / f'{date}-segment-colors.json').read_text(encoding='utf-8'))

    magnitudes = np.linalg.norm(raw_preds, axis=1)
    changes = np.concatenate([[0.0], np.linalg.norm(np.diff(raw_preds, axis=0), axis=1)])
    text_density = np.array([
        len(segment['tts_text']) / max(len(segment['lines']), 1)
        for segment in prepared_segments
    ], dtype='float32')

    attention = normalize(magnitudes * 0.65 + changes * 0.35)
    load = normalize(text_density * 0.55 + magnitudes * 0.45)
    novelty = normalize(changes)

    frontend_segments = []
    for idx, segment in enumerate(prepared_segments):
        att = round(float(attention[idx]), 1)
        lod = round(float(load[idx]), 1)
        nov = round(float(novelty[idx]), 1)
        frontend_segments.append({
            'segment_id': segment['segment_id'],
            'start_time': segment['start_time'],
            'end_time': segment['end_time'],
            'proxies': {
                'attention_proxy': att,
                'load_proxy': lod,
                'novelty_proxy': nov,
            },
            'labels': labels_for(att, lod, nov),
            'interpretation': interpretation_for(att, lod, nov),
        })

    strongest = sorted(frontend_segments, key=lambda x: x['proxies']['attention_proxy'], reverse=True)[:2]
    risky = sorted(frontend_segments, key=lambda x: x['proxies']['load_proxy'] - x['proxies']['attention_proxy'] * 0.3, reverse=True)[:2]

    personas = [
        {
            'persona_id': 'novice',
            'label': '초보 수강자',
            'overall_score': round(float(np.mean(attention * 0.55 + (100 - load) * 0.45)), 1),
            'top_positive_segment_ids': [seg['segment_id'] for seg in strongest],
            'top_risk_segment_ids': [seg['segment_id'] for seg in risky],
            'reaction_summary': '기본 개념 흐름이 선명한 구간에서 반응이 올라가고, 밀도가 높아지면 따라가기 어려워질 수 있습니다.',
        },
        {
            'persona_id': 'builder',
            'label': '실습형 수강자',
            'overall_score': round(float(np.mean(attention * 0.35 + novelty * 0.65)), 1),
            'top_positive_segment_ids': [seg['segment_id'] for seg in strongest],
            'top_risk_segment_ids': [seg['segment_id'] for seg in risky],
            'reaction_summary': '새로운 예시나 구조 전환에서 반응이 커지고, 반복 설명만 이어지면 체감 가치가 줄 수 있습니다.',
        },
        {
            'persona_id': 'pace_sensitive',
            'label': '속도 민감형',
            'overall_score': round(float(np.mean((100 - np.abs(load - 50)) * 0.6 + attention * 0.4)), 1),
            'top_positive_segment_ids': [seg['segment_id'] for seg in strongest],
            'top_risk_segment_ids': [seg['segment_id'] for seg in risky],
            'reaction_summary': '속도와 정보량이 맞는 구간에서 반응이 안정적이며, 부하가 높아지면 회복이 늦어질 수 있습니다.',
        },
    ]

    frontend_payload = {
        'lecture_date': date,
        'source_model': 'tribev2',
        'source_modality': 'text_tts',
        'generated_at': '2026-03-30T12:00:00+09:00',
        'assets': {
            'mesh_glb': '/data/simulations/brain-mesh.glb',
            'segment_colors_json': f'/data/simulations/{date}-segment-colors.json',
        },
        'metadata': {
            'subject': metadata_by_date[date]['subject'],
            'content': metadata_by_date[date]['content'],
            'instructor': metadata_by_date[date]['instructor'],
            'segment_minutes': 5,
        },
        'lecture_summary': {
            'strongest_segment_ids': [seg['segment_id'] for seg in strongest],
            'risk_segment_ids': [seg['segment_id'] for seg in risky],
            'summary_text': 'TRIBE raw output의 세그먼트별 반응 강도와 변화율을 attention/load/novelty 프록시로 축약한 결과입니다.',
            'caution_text': 'TRIBE v2 기반 신경 반응 프록시 시뮬레이션이며 실제 수강생 감정·설문을 대체하지 않습니다.',
        },
        'segments': frontend_segments,
        'personas': personas,
    }

    (FRONTEND_DIR / f'{date}.json').write_text(json.dumps(frontend_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    (FRONTEND_DIR / f'{date}-transcript.json').write_text(json.dumps(transcript_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print('frontend payload ready', date, len(color_payload['segments']))
